In [1]:
%pip install requests
import requests
import pandas as pd
import debugpy 

Note: you may need to restart the kernel to use updated packages.


# Rainfall

In [ ]:

from datetime import datetime, timedelta

def create_dates(start_date, end_date):
    # Create a date range
    
    debugpy.breakpoint() 
    dates = pd.date_range(start=start_date, end=end_date)
    # Convert to list of strings
    date_list = dates.strftime("%Y-%m-%d").tolist()
    return date_list

def create_timestamps(dates):
    timestamps = []
    for d in dates:
        dt = datetime.strptime(d, "%Y-%m-%d")
        for h in range(24):
            ts = dt + timedelta(hours=h)
            timestamps.append(ts.isoformat())  # ISO 8601 format
    return timestamps

In [3]:
import time
import debugpy

# creates the stations df
def get_stations(timestamps, url):
    # get stations
    params = {'date': timestamps[0]}
    response = requests.get(url, params=params)
    try:
        response.raise_for_status()  # catches non-200 HTTP responses
        data = response.json()       # parse once, reuse

        if data.get('code') == 0:
            stations_df = pd.json_normalize(data["data"],record_path="stations",sep=".")
            stations_df.rename(
                columns={
                    'location.latitude': 'latitude',
                    'location.longitude': 'longitude'
                },
                inplace=True
            )
            time.sleep(2)
            return stations_df

        else:
            print(f"API returned error code: {data.get('code')} at timestamp {params}")

    except ValueError:
        # JSON decode error
        print("Invalid JSON response:", response.text)

    except KeyError:
        # Missing expected fields
        print("Missing expected fields in JSON:", data)

    except requests.exceptions.RequestException as e:
        # network errors, timeout, DNS failure etc.
        print("Request failed:", e)

# creates the readings df
def get_readings(timestamps, url):
    count = 1
    readings_df_list = []
    total = len(timestamps)
    for timestamp in timestamps:
        print(f"Pulling {count}/{total} — {timestamp}")
        params = {"date": timestamp}
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()      # catch non-200 codes
            debugpy.breakpoint()
            data = response.json()

            # API-level errors
            if data.get("code") != 0:
                print(f"API returned error code {data.get('code')} for {timestamp}")
                count += 1
                time.sleep(2)
                continue

            # Extract readings safely
            readings = pd.json_normalize(data["data"]["readings"],record_path="data", meta="timestamp",sep=".",errors="ignore")
            readings['query_timestamp'] = timestamp
            readings_df_list.append(readings)

        except requests.exceptions.Timeout:
            print(f"Timeout at {timestamp}, retrying...")
            time.sleep(3)
            continue

        except requests.exceptions.RequestException as e:
            print(f"HTTP error at {timestamp}: {e}")
            continue
        except (KeyError, ValueError) as e:
            print(f"JSON structure error at {timestamp}: {e}")
            continue

        count += 1
        time.sleep(2)  # avoid rate limits

    readings_df = pd.concat(readings_df_list)

    return readings_df

# calls the api to generate the raw data dfs
def get_data(timestamps):
    url = 'https://api-open.data.gov.sg/v2/real-time/api/rainfall'
    print(f'Estimated time to pull: {(len(timestamps))*3 + 2} seconds or {((len(timestamps))*3 + 2)/60} minutes')
    
    stations_df = get_stations(timestamps, url)
    readings_df = get_readings(timestamps, url)
    debugpy.breakpoint() 
    print('rainfall api pull complete')
    return stations_df, readings_df

In [4]:
start_date = '2025-01-01'
end_date = '2025-01-02'

dates = create_dates(start_date=start_date, end_date=end_date)


In [ ]:
timestamps = create_timestamps(dates)
stations_df, readings_df = get_data(timestamps)

Estimated time to pull: 146 seconds or 2.433333333333333 minutes
Pulling 1/48 — 2025-01-01T00:00:00
Pulling 2/48 — 2025-01-01T01:00:00


In [ ]:
stations_df.to_csv('rainfall/rain_stations.csv', index=False)
stations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         60 non-null     object 
 1   deviceId   60 non-null     object 
 2   name       60 non-null     object 
 3   latitude   60 non-null     float64
 4   longitude  60 non-null     float64
dtypes: float64(2), object(3)
memory usage: 2.5+ KB


In [ ]:
filename=f'rainfall/rain_readings_{start_date}_to_{end_date}.csv'
readings_df.to_csv(filename, index=False)
readings_df

,stationId,value,timestamp,query_timestamp
0,S218,0.0,2025-02-01T00:00:00+08:00,2025-02-01T00:00:00
1,S219,0.0,2025-02-01T00:00:00+08:00,2025-02-01T00:00:00
2,S216,0.0,2025-02-01T00:00:00+08:00,2025-02-01T00:00:00
3,S217,0.0,2025-02-01T00:00:00+08:00,2025-02-01T00:00:00
4,S214,0.0,2025-02-01T00:00:00+08:00,2025-02-01T00:00:00
...,...,...,...,...
54,S79,0.0,2025-11-30T23:00:00+08:00,2025-11-30T23:00:00
55,S78,0.0,2025-11-30T23:00:00+08:00,2025-11-30T23:00:00
56,S111,0.0,2025-11-30T23:00:00+08:00,2025-11-30T23:00:00
57,S112,0.0,2025-11-30T23:00:00+08:00,2025-11-30T23:00:00
